In [1]:
import base64
import json
import os
import pickle
import tempfile
from pathlib import Path

import yaml
import torch
from feluda import Feluda
from tqdm.notebook import tqdm
from transformers import AutoProcessor, CLIPModel

In [2]:
%%time
%%capture
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
model.to(device)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CPU times: user 700 ms, sys: 298 ms, total: 997 ms
Wall time: 4.97 s


In [3]:
# Instead of loading from a file, we'll define the configuration directly in code
def create_feluda_config():
    """Create and return a configuration dictionary for Feluda"""
    config = {
        "operators": {
            "label": "Operators",
            "parameters": [
                {
                    "name": "Dimension Reduction",
                    "type": "dimension_reduction",
                    "parameters": {"index_name": "video"},
                },
                {
                    "name": "Cluster Embeddings",
                    "type": "cluster_embeddings",
                    "parameters": {"index_name": "video"},
                },
            ],
        }
    }

    # Create a temporary file to store the configuration
    fd, config_path = tempfile.mkstemp(suffix=".yml")
    with open(fd, "w") as f:
        yaml.dump(config, f)

    return config_path


config_path = create_feluda_config()
print(f"Config created at {config_path}")

Config created at /tmp/tmpek_4oyp6.yml


In [4]:
feluda = Feluda(config_path)
feluda.setup()

dimension_reduction_operator = feluda.operators.get()["dimension_reduction"]
cluster_operator = feluda.operators.get()["cluster_embeddings"]
os.remove(config_path)

t-SNE model successfully initialized


In [5]:
with open("video_data_2.pkl", "rb") as f:
    video_data = pickle.load(f)

print(len(video_data))

1472


In [6]:
operator_parameters = []
for item in tqdm(video_data):
    operator_parameters.append(
        {
            "payload": {
                "video_id": item["video_id"],
                "video_path": item["video_path"],
                "thumbnail_path": item["thumbnail_path"],
            },
            "embedding": item["embedding"],
        }
    )

print(f"Successfully processed {len(operator_parameters)} videos")

  0%|          | 0/1472 [00:00<?, ?it/s]

Successfully processed 1472 videos


In [7]:
operator_parameters[0].keys()

dict_keys(['payload', 'embedding'])

In [8]:
%%time
# Reduce dimensions using operator for plotting t-SNE embeddings
data = dimension_reduction_operator.run(operator_parameters)

CPU times: user 17.8 s, sys: 23.4 ms, total: 17.8 s
Wall time: 4.32 s


In [9]:
data[0]

{'payload': {'video_id': 'DI3VngZvkSu',
  'video_path': '/home/aatman/Aatman/Tattle/tattle-research/brainrot/main_analysis_2/visualisation/video_reels/DI3VngZvkSu.mp4',
  'thumbnail_path': 'thumbnails/DI3VngZvkSu.jpg'},
 'reduced_embedding': [24.375722885131836, 10.035624504089355]}

In [10]:
# now lets find the range of x,y 2D coords
xs = []
ys = []

for item in data:
    x, y = item["reduced_embedding"]
    xs.append(x)
    ys.append(y)

x_min, x_max = min(xs), max(xs)
y_min, y_max = min(ys), max(ys)

print(f"X range: min={x_min:.2f}, max={x_max:.2f}")
print(f"Y range: min={y_min:.2f}, max={y_max:.2f}")

X range: min=-44.02, max=31.00
Y range: min=-26.31, max=27.70


## add day

In [11]:
for item in data:
    video_filename = os.path.basename(item["payload"]["video_path"])
    video_id = os.path.splitext(video_filename)[0]
    item["payload"]["id"] = video_id

In [12]:
# Base directory where day1, day2, day3 folders exist
base_dir = Path("/home/aatman/Aatman/Tattle/tattle-research/brainrot/instagram_scrapper/denny_reels_download")

# Build a mapping from video ID to day number
id_to_day = {}
for day_folder in base_dir.iterdir():
    if day_folder.is_dir() and day_folder.name.startswith("day"):
        day_number = int(day_folder.name.replace("day", ""))
        for mp4_file in day_folder.glob("*.mp4"):
            video_id = mp4_file.stem  # removes '.mp4'
            id_to_day[video_id] = day_number

# Now update your data payloads
for item in data:
    video_id = item["payload"]["id"]
    item["payload"]["day"] = id_to_day.get(video_id, None)

In [13]:
data[0]

{'payload': {'video_id': 'DI3VngZvkSu',
  'video_path': '/home/aatman/Aatman/Tattle/tattle-research/brainrot/main_analysis_2/visualisation/video_reels/DI3VngZvkSu.mp4',
  'thumbnail_path': 'thumbnails/DI3VngZvkSu.jpg',
  'id': 'DI3VngZvkSu',
  'day': 4},
 'reduced_embedding': [24.375722885131836, 10.035624504089355]}

### classify zero shot

In [14]:
for d, o in zip(data, operator_parameters):
    assert os.path.basename(d["payload"]["video_path"]) == os.path.basename(o["payload"]["video_path"])

In [15]:
def run(embedding, labels):
    """
    Runs zero-shot classification on a stored embedding.
    """
    inputs = processor(text=labels, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}  # Move to device

    with torch.no_grad():
        text_features = model.get_text_features(**inputs)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        video_embedding = torch.tensor(embedding).to(device)
        video_embedding = video_embedding / video_embedding.norm(dim=-1, keepdim=True)

        similarity = (video_embedding @ text_features.T).softmax(dim=-1)

    return {
        "prediction": labels[similarity.argmax().item()],
        "probs": similarity.cpu().numpy().tolist(),
    }

In [16]:
# labels = ["cooking", "dancing", "podcasts", "animals", "screen capture", 
#           "talking head", "standup comedy", "monologue", "movies", "miscellaneous", "shopping"]
labels = ["cooking", "dancing", "podcasts", "animals", "screen capture", "talking head", "standup comedy", "monologue", "miscellaneous", 
          "shopping", "travel", "fitness", "adventure", "home decor", "water sports", "DIY"]

In [17]:
%%time
for item, op in tqdm(zip(data, operator_parameters), total=len(data), desc="Running zero-shot classification"):
    embedding = op["embedding"]
    result = run(embedding, labels)
    item["payload"]["zero_shot_prediction"] = result["prediction"]

Running zero-shot classification:   0%|          | 0/1472 [00:00<?, ?it/s]

CPU times: user 4min 53s, sys: 1.59 s, total: 4min 54s
Wall time: 1min 13s


### clustering

In [18]:
cluster_results = cluster_operator.run(operator_parameters, n_clusters=None, modality="video")

In [19]:
print(f"Total number of clusters: {len(cluster_results)}")
print(f"Cluster Names: {cluster_results.keys()}")

Total number of clusters: 55
Cluster Names: dict_keys(['cluster_22', 'cluster_15', 'cluster_44', 'cluster_19', 'cluster_24', 'cluster_33', 'cluster_9', 'cluster_47', 'cluster_7', 'cluster_53', 'cluster_25', 'cluster_20', 'cluster_0', 'cluster_10', 'cluster_51', 'cluster_27', 'cluster_49', 'cluster_37', 'cluster_17', 'cluster_11', 'cluster_26', 'cluster_14', 'cluster_38', 'cluster_42', 'cluster_2', 'cluster_36', 'cluster_50', 'cluster_23', 'cluster_28', 'cluster_45', 'cluster_1', 'cluster_54', 'cluster_29', 'cluster_43', 'cluster_4', 'cluster_18', 'cluster_31', 'cluster_8', 'cluster_16', 'cluster_52', 'cluster_5', 'cluster_6', 'cluster_40', 'cluster_3', 'cluster_48', 'cluster_34', 'cluster_13', 'cluster_41', 'cluster_39', 'cluster_32', 'cluster_12', 'cluster_21', 'cluster_30', 'cluster_35', 'cluster_46'])


In [20]:
# Create a mapping from video path to cluster label
video_to_cluster = {}
for cluster_name, items in cluster_results.items():
    cluster_label = cluster_name.split('_')[1]  # Extract the label number from "cluster_X"
    for item in items:
        video_path = item["video_path"]
        video_to_cluster[video_path] = cluster_label

print(len(video_to_cluster))

1472


### normalisation

In [21]:
# Min-max scaling to 0-1 range
normalized_data = []
for item in data:
    x, y = item["reduced_embedding"]

    # Normalize to 0-1 range
    x_norm = (x - x_min) / (x_max - x_min)
    y_norm = (y - y_min) / (y_max - y_min)

    normalized_item = item.copy()
    normalized_item["reduced_embedding_normalized"] = [x_norm, y_norm]
    normalized_data.append(normalized_item)

In [22]:
normalized_data[0]

{'payload': {'video_id': 'DI3VngZvkSu',
  'video_path': '/home/aatman/Aatman/Tattle/tattle-research/brainrot/main_analysis_2/visualisation/video_reels/DI3VngZvkSu.mp4',
  'thumbnail_path': 'thumbnails/DI3VngZvkSu.jpg',
  'id': 'DI3VngZvkSu',
  'day': 4,
  'zero_shot_prediction': 'screen capture'},
 'reduced_embedding': [24.375722885131836, 10.035624504089355],
 'reduced_embedding_normalized': [0.9117235842666586, 0.672933235498815]}

In [23]:
# # Min-max scaling to custom range (e.g., for SVG coordinates)
# width, height = 800, 600  # Your desired visualization dimensions
# padding = 50  # Padding from edges

# normalized_data = []
# for item in data:
#     x, y = item["reduced_embedding"]

#     # Normalize to fit within width/height with padding
#     x_scaled = padding + (x - x_min) / (x_max - x_min) * (width - 2 * padding)
#     y_scaled = padding + (y - y_min) / (y_max - y_min) * (height - 2 * padding)

#     normalized_item = item.copy()
#     normalized_item["reduced_embedding"] = [x_scaled, y_scaled]
#     normalized_data.append(normalized_item)

In [24]:
# try:
#     with open(thumbnail_path, "rb") as img_file:
#         thumbnail_bytes = img_file.read()
#         thumbnail_base64 = base64.b64encode(thumbnail_bytes).decode("utf-8")
# except FileNotFoundError:
#     print(f"Thumbnail not found: {thumbnail_path}")
#     thumbnail_base64 = None

# # Read video and encode to base64
# try:
#     with open(video_path, "rb") as vid_file:
#         video_bytes = vid_file.read()
#         video_base64 = base64.b64encode(video_bytes).decode("utf-8")
# except FileNotFoundError:
#     print(f"Video not found: {video_path}")
#     video_base64 = None

In [25]:
enriched_data = []

for item in tqdm(normalized_data, desc="Encoding thumbnails and videos"):
    video_path = item["payload"]["video_path"]
    video_id = os.path.splitext(os.path.basename(video_path))[0]
    cluster_label = video_to_cluster.get(video_path, "unknown")

    # Read thumbnail and encode to base64
    thumbnail_path = item["payload"]["thumbnail_path"]

    enriched_payload = item["payload"].copy()
    enriched_payload["video_id"] = video_id
    # enriched_payload["thumbnail_base64"] = thumbnail_base64
    # enriched_payload["video_base64"] = video_base64
    enriched_payload["cluster"] = cluster_label

    enriched_item = item.copy()
    enriched_item["payload"] = enriched_payload
    enriched_item["cluster"] = cluster_label
    enriched_data.append(enriched_item)

Encoding thumbnails and videos:   0%|          | 0/1472 [00:00<?, ?it/s]

In [26]:
# Print some statistics about clustering
cluster_counts = {}
for item in enriched_data:
    cluster = item["cluster"]
    if cluster not in cluster_counts:
        cluster_counts[cluster] = 0
    cluster_counts[cluster] += 1

print("Cluster distribution:")
for cluster, count in sorted(cluster_counts.items()):
    print(f"Cluster {cluster}: {count} videos")

Cluster distribution:
Cluster 0: 17 videos
Cluster 1: 2 videos
Cluster 10: 51 videos
Cluster 11: 29 videos
Cluster 12: 1 videos
Cluster 13: 9 videos
Cluster 14: 16 videos
Cluster 15: 65 videos
Cluster 16: 3 videos
Cluster 17: 14 videos
Cluster 18: 39 videos
Cluster 19: 54 videos
Cluster 2: 20 videos
Cluster 20: 52 videos
Cluster 21: 1 videos
Cluster 22: 10 videos
Cluster 23: 10 videos
Cluster 24: 59 videos
Cluster 25: 39 videos
Cluster 26: 41 videos
Cluster 27: 29 videos
Cluster 28: 28 videos
Cluster 29: 11 videos
Cluster 3: 8 videos
Cluster 30: 1 videos
Cluster 31: 21 videos
Cluster 32: 4 videos
Cluster 33: 29 videos
Cluster 34: 13 videos
Cluster 35: 1 videos
Cluster 36: 53 videos
Cluster 37: 49 videos
Cluster 38: 32 videos
Cluster 39: 22 videos
Cluster 4: 25 videos
Cluster 40: 4 videos
Cluster 41: 10 videos
Cluster 42: 25 videos
Cluster 43: 34 videos
Cluster 44: 61 videos
Cluster 45: 10 videos
Cluster 46: 1 videos
Cluster 47: 27 videos
Cluster 48: 3 videos
Cluster 49: 40 videos
Clust

In [27]:
with open("video_tsne_enriched_data_viz_without_base64.json", "w") as f:
    json.dump(enriched_data, f, indent=2)

In [28]:
# labels = ["cooking", "dancing", "podcasts", "pets/animals", "movie scenes", "self monologues"]
# # brainrot
# labels = ["food", "talking head", "brainrot", "animals"]
# labels = ["cooking", "dancing", "podcasts", "animals", "screen capture", "talking head"]